# Proof of Concept

Calculate a simple forward pass from 3DGS to 3DGS

In [6]:
import torch
import torch.nn as nn

class POC_Network(nn.Module):
    def __init__(self, dim, bottleneck_size):
        super(POC_Network, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(dim, 32),
            nn.ReLU(),
            nn.Linear(32, bottleneck_size),
            nn.ReLU(),
            nn.Linear(bottleneck_size, 32),
            nn.ReLU(),
            nn.Linear(32, dim)
        )

    def forward(self, x):
        return self.model(x)


In [7]:
EPOCHS = 1000
LR = 1e-3
BOTTLENECK_SIZES = [10, 8, 6,4]
SAMPLES_LIST = [1, 2, 5, 10]


In [8]:
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

In [ ]:
import wandb
import torch.nn.functional as F
from torch.optim import AdamW
from dataset import get_dataloader

def train_run(bottleneck_size, num_samples):
    network = POC_Network(dim=14, bottleneck_size=bottleneck_size).to(DEVICE)
    loader = get_dataloader("./data/uco3d_data_raw", batch_size=1, num_samples=num_samples)
    optim = AdamW(network.parameters(), lr=LR)

    run = wandb.init(
        entity="leonardkuech-technical-university-of-munich",
        project="feed-forward radiance field conversion",
        config={
            "learning_rate": LR,
            "bottleneck_size": bottleneck_size,
            "dataset": "uco3d",
            "epochs": EPOCHS,
            "samples": num_samples,
            "device": str(DEVICE),
        },
        reinit=True,
    )

    for epoch in range(EPOCHS):
        optim.zero_grad()
        batch = next(iter(loader))
        x = batch[0]["splat_tensor"].to(DEVICE)
        output = network(x)
        loss = F.mse_loss(x, output)
        loss.backward()
        optim.step()
        run.log({"loss": loss.item()})

    run.finish()


In [10]:
for bottleneck_size in BOTTLENECK_SIZES:
    for num_samples in SAMPLES_LIST:
        print(f"Training: bottleneck={bottleneck_size}, samples={num_samples}")
        train_run(bottleneck_size, num_samples)


Training: bottleneck=10, samples=1


wandb: Currently logged in as: leonardkuech (leonardkuech-technical-university-of-munich) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x1326dbed0>> (for post_run_cell), with arguments args (<ExecutionResult object at 1111be890, execution_count=10 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 132380550, raw_cell="for bottleneck_size in BOTTLENECK_SIZES:
    for n.." transformed_cell="for bottleneck_size in BOTTLENECK_SIZES:
    for n.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Users/leonardkuch/Documents/Universitaet/Master/IS/ADL4CV/radiance-field-project/poc_network.ipynb#W5sZmlsZQ%3D%3D> result=None>,),kwargs {}:


ConnectionResetError: Connection lost